# Copa do Mundo 2026 — informações gerais (API-Football v3)

Este notebook usa o cliente já existente em `api-football/authentication.py` e variáveis de `api-football/.env` (ver `.env.example`).

**Documentação:** [API-Football v3](https://www.api-football.com/documentation-v3)  
**Referência de dados:** a FIFA World Cup na API-Football v3 usa **`league=1`**. Para **metadados da edição 2026** (datas oficiais da API, cobertura, `current`) usamos **`GET /leagues?id=1`** e filtramos a entrada com **`year: 2026`** no array `seasons` — este pedido **sem** `season` costuma funcionar também em planos gratuitos. Endpoints com **`season=2026`** (`fixtures`, `teams`, `fixtures/rounds`, etc.) podem exigir **plano pago**; o notebook mostra o erro da API quando isso acontecer.

Execute as células em ordem: confirmação da chave; resumo da liga e da temporada 2026; depois rondas, equipas e jogos para `season=2026` quando o plano permitir.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

# Pasta do provedor: raiz do repo, `api-football/` ou `api-football/notebooks/` (cwd do Jupyter)
# Se o cwd do processo apontar para uma pasta já removida/renomeada, `Path.cwd()` levanta FileNotFoundError.
def _resolve_notebook_base() -> Path:
    try:
        return Path.cwd().resolve()
    except FileNotFoundError:
        pass
    # Depois de cwd inválido: preferir IPython/Jupyter antes de `PWD`.
    # `os.path.isdir(os.environ["PWD"])` pode bloquear muito tempo se `PWD` for rede/mount lento.
    try:
        from IPython import get_ipython

        sh = get_ipython()
        sd = getattr(sh, "starting_dir", None) if sh else None
        if sd and os.path.isdir(sd):
            return Path(sd).resolve()
    except Exception:
        pass
    for key in ("JUPYTER_SERVER_ROOT", "PWD"):
        raw = os.environ.get(key)
        if raw and os.path.isdir(raw):
            return Path(raw).resolve()
    raise RuntimeError(
        "O diretório de trabalho do kernel já não existe no disco (cwd inválido). "
        "Reinicia o kernel do Jupyter e abre o notebook a partir da pasta do repositório."
    ) from None


_nb_dir = _resolve_notebook_base()

def _find_api_football_provider(base: Path) -> Path:
    for c in (
        base / "api-football",
        base.parent if base.name == "notebooks" else None,
        base,
    ):
        if c is None:
            continue
        if (c / "authentication.py").is_file() and (c / "env_loader.py").is_file():
            return c.resolve()
    raise RuntimeError(
        "Não encontrei api-football/ (authentication.py). "
        f"cwd={base!s}. Usa cwd na raiz do repositório ou em api-football/notebooks."
    )


_provider = _find_api_football_provider(_nb_dir)
if str(_provider) not in sys.path:
    sys.path.insert(0, str(_provider))

# Carregar `.env` antes de ler `os.environ` (evita depender só do `load_local_env` dentro de `get_json`)
from env_loader import load_local_env

load_local_env()

BASE = os.environ.get("API_FOOTBALL_BASE_URL", "").rstrip("/")
LEAGUE_WORLD_CUP = 1
SEASON = os.environ.get("SEASON", "2026")

print(f"BASE: {BASE}")
print(f"LEAGUE_WORLD_CUP: {LEAGUE_WORLD_CUP}")
print(f"SEASON: {SEASON}")


BASE: https://v3.football.api-sports.io
LEAGUE_WORLD_CUP: 1
SEASON: 2022


In [2]:

from authentication import get_json, verify_api_key

if not BASE:
    raise RuntimeError(
        "Defina API_FOOTBALL_BASE_URL em api-football/.env (ex.: https://v3.football.api-sports.io)"
    )



def _api_errors(data: dict) -> dict | list | str | None:
    err = data.get("errors")
    if err is None or err == [] or err == {}:
        return None
    return err


def _print_plan_hint(err: object) -> None:
    text = str(err).lower()
    if "plan" in text or "free plans" in text:
        print(
            "\n(i) Plano sem acesso a `season=2026` nestes endpoints. "
            "Na secção 1, `GET /leagues?id=1` (sem season) costuma trazer datas e cobertura da 2026 em `seasons`."
        )


print("Base URL:", BASE)
check = verify_api_key()
print("Chave válida:", check.ok, "| HTTP:", check.http_status)
if not check.ok:
    print(json.dumps(check.data, indent=2, ensure_ascii=False))

Base URL: https://v3.football.api-sports.io
Chave válida: True | HTTP: 200


## 1. Liga e temporada 2026 (datas + cobertura)

**`GET /leagues?id=1`** devolve a competição **World Cup** e **todas** as temporadas conhecidas; localizamos **`year == 2026`** para ver `start`, `end`, `current` e **`coverage`** (o que o plano/API expõem para jogos, classificações, jogadores, etc.).

Opcionalmente, **`GET /leagues?id=1&season=2026`** devolve só essa temporada — em alguns planos gratuitos a API responde com erro de plano; nesse caso a informação geral da 2026 continua disponível no pedido **sem** `season`, acima.

In [3]:
status, data = get_json(f"leagues?id={LEAGUE_WORLD_CUP}")
print("HTTP:", status)
err = _api_errors(data)
if err:
    print(json.dumps(data, indent=2, ensure_ascii=False))
else:
    resp = (data.get("response") or [{}])[0]
    league = resp.get("league") or {}
    country = resp.get("country") or {}
    seasons = resp.get("seasons") or []
    season_2026 = next((s for s in seasons if s.get("year") == SEASON), None)

    print("\n=== Competição ===")
    print("Nome:", league.get("name"), "| Tipo:", league.get("type"))
    print("ID liga:", league.get("id"), "| Logo:", league.get("logo"))
    print("\n=== País / âmbito ===")
    print("País:", country.get("name"), "| Código:", country.get("code"))

    print("\n=== Temporada", SEASON, "===")
    if not season_2026:
        print("Entrada 2026 não encontrada em `seasons`. Lista de anos:", [s.get("year") for s in seasons])
    else:
        print("Início (API):", season_2026.get("start"), "| Fim:", season_2026.get("end"))
        print("Marcada como atual:", season_2026.get("current"))
        print("\nCobertura (campos que a API declara para esta edição):")
        print(json.dumps(season_2026.get("coverage") or {}, indent=2, ensure_ascii=False))

    print("\n--- Opcional: mesmo recurso filtrado por season (pode falhar em plano Free) ---")
    s2, d2 = get_json(f"leagues?id={LEAGUE_WORLD_CUP}&season={SEASON}")
    print("HTTP:", s2, "| errors:", _api_errors(d2))
    if _api_errors(d2):
        _print_plan_hint(_api_errors(d2))
        print(json.dumps(d2.get("errors"), indent=2, ensure_ascii=False))

HTTP: 200

=== Competição ===
Nome: World Cup | Tipo: Cup
ID liga: 1 | Logo: https://media.api-sports.io/football/leagues/1.png

=== País / âmbito ===
País: World | Código: None

=== Temporada 2022 ===
Entrada 2026 não encontrada em `seasons`. Lista de anos: [2010, 2014, 2018, 2022, 2026]

--- Opcional: mesmo recurso filtrado por season (pode falhar em plano Free) ---
HTTP: 200 | errors: None


## 2. Rondas / fases da competição

**`/fixtures/rounds`** lista as rondas disponíveis para `league` e `season` (útil para perceber fases antes de filtrar jogos).

In [4]:
status, data = get_json(
    f"fixtures/rounds?league={LEAGUE_WORLD_CUP}&season={SEASON}"
)
print("HTTP:", status)
if _api_errors(data):
    _print_plan_hint(_api_errors(data))
    print(json.dumps(data, indent=2, ensure_ascii=False))
else:
    rounds = data.get("response") or []
    print("Total de rondas/fases:", len(rounds))
    for i, r in enumerate(rounds[:20]):
        print(f"  {i+1}. {r}")
    if len(rounds) > 20:
        print("  ...")

    print("\n(JSON completo da lista)")
    print(json.dumps(rounds, indent=2, ensure_ascii=False))

HTTP: 200
Total de rondas/fases: 8
  1. Group Stage - 1
  2. Group Stage - 2
  3. Group Stage - 3
  4. Round of 16
  5. Quarter-finals
  6. Semi-finals
  7. 3rd Place Final
  8. Final

(JSON completo da lista)
[
  "Group Stage - 1",
  "Group Stage - 2",
  "Group Stage - 3",
  "Round of 16",
  "Quarter-finals",
  "Semi-finals",
  "3rd Place Final",
  "Final"
]


## 3. Equipas participantes

**`/teams`** com `league` e `season` devolve as seleções com IDs, nomes e logos (consoante a API já tenha a lista completa para 2026).

In [5]:
status, data = get_json(f"teams?league={LEAGUE_WORLD_CUP}&season={SEASON}")
print("HTTP:", status)
if _api_errors(data):
    _print_plan_hint(_api_errors(data))
    print(json.dumps(data, indent=2, ensure_ascii=False))
else:
    teams = data.get("response") or []
    print("Entradas na resposta:", len(teams))
    for row in teams[:15]:
        t = row.get("team") or {}
        print(" -", t.get("id"), t.get("name"))
    if len(teams) > 15:
        print(" ...")

    print("\n(Primeiro item completo, para inspeção de campos)")
    if teams:
        print(json.dumps(teams[0], indent=2, ensure_ascii=False))

HTTP: 200
Entradas na resposta: 32
 - 1 Belgium
 - 2 France
 - 3 Croatia
 - 6 Brazil
 - 7 Uruguay
 - 9 Spain
 - 10 England
 - 12 Japan
 - 13 Senegal
 - 14 Serbia
 - 15 Switzerland
 - 16 Mexico
 - 17 South Korea
 - 20 Australia
 - 21 Denmark
 ...

(Primeiro item completo, para inspeção de campos)
{
  "team": {
    "id": 1,
    "name": "Belgium",
    "code": "BEL",
    "country": "Belgium",
    "founded": 1895,
    "national": true,
    "logo": "https://media.api-sports.io/football/teams/1.png"
  },
  "venue": {
    "id": 173,
    "name": "Stade Roi Baudouin",
    "address": "Avenue de Marathon 135/2, Laken",
    "city": "Brussel",
    "capacity": 50093,
    "surface": "grass",
    "image": "https://media.api-sports.io/football/venues/173.png"
  }
}


## 4. Resumo de jogos (contagem e amostra)

**`/fixtures`** com `league` e `season` lista os jogos; aqui só contamos e mostramos os primeiros para não sobrecarregar o output.

In [6]:
status, data = get_json(
    f"fixtures?league={LEAGUE_WORLD_CUP}&season={SEASON}"
)
print("HTTP:", status)
if _api_errors(data):
    _print_plan_hint(_api_errors(data))
    print(json.dumps(data, indent=2, ensure_ascii=False))
else:
    fixtures = data.get("response") or []
    print("Total de fixtures devolvidos neste pedido:", len(fixtures))
    paging = data.get("paging") or {}
    if paging:
        print("Paging:", paging)

    print("\nPrimeiros 5 jogos (data / equipas / estado):")
    for fx in fixtures[:5]:
        fix = fx.get("fixture") or {}
        teams = fx.get("teams") or {}
        home = (teams.get("home") or {}).get("name")
        away = (teams.get("away") or {}).get("name")
        ts = fix.get("date")
        st = (fix.get("status") or {}).get("long")
        print(f"  {ts} | {home} vs {away} | {st}")

    if fixtures:
        print("\n(Primeiro fixture completo — campos úteis para o dashboard)")
        print(json.dumps(fixtures[0], indent=2, ensure_ascii=False))

HTTP: 200
Total de fixtures devolvidos neste pedido: 64
Paging: {'current': 1, 'total': 1}

Primeiros 5 jogos (data / equipas / estado):
  2022-11-20T16:00:00+00:00 | Qatar vs Ecuador | Match Finished
  2022-11-21T13:00:00+00:00 | England vs Iran | Match Finished
  2022-11-21T16:00:00+00:00 | Senegal vs Netherlands | Match Finished
  2022-11-21T19:00:00+00:00 | USA vs Wales | Match Finished
  2022-11-22T10:00:00+00:00 | Argentina vs Saudi Arabia | Match Finished

(Primeiro fixture completo — campos úteis para o dashboard)
{
  "fixture": {
    "id": 855736,
    "referee": "D. Orsato",
    "timezone": "UTC",
    "date": "2022-11-20T16:00:00+00:00",
    "timestamp": 1668960000,
    "periods": {
      "first": 1668960000,
      "second": 1668963600
    },
    "venue": {
      "id": null,
      "name": "Al Bayt Stadium",
      "city": "Al Khor"
    },
    "status": {
      "long": "Match Finished",
      "short": "FT",
      "elapsed": 90,
      "extra": null
    }
  },
  "league": {
    "i